# DocsMind lab: turn BRISKODA posts into production retrieval chunks

This notebook covers one pipeline boundary:

**Crawl → Validate → Profile → Chunk → Inspect → Export**

It deliberately stops before embedding. The goal is to verify what one vector will represent before spending GPU time or AWS credits.

The raw crawler record is not automatically the correct retrieval unit. A normal post can remain whole, a tiny reply may need the previous post, and a long repair guide needs sentence-aware splitting.


## What you should learn

By the end, you should be able to explain:

- why thread length and post length are different problems;
- why a long-context embedding model does not remove the need for chunking;
- why short replies borrow limited conversational context;
- how stable IDs prevent duplicate vectors;
- how the same chunks can feed local FAISS or AWS OpenSearch;
- what must be measured before choosing GTE ModernBERT or Qwen3 Embedding.

Run the cells from top to bottom. The notebook writes only derived experiment files. It does not modify the crawler corpus.


In [ ]:
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import random
import shutil
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'docsmind').exists():
    ROOT = ROOT.parent
if not (ROOT / 'docsmind').exists():
    raise RuntimeError('Start Jupyter from the DocsMind repository.')
sys.path.insert(0, str(ROOT))

from docsmind.ingestion.briskoda_chunks import (
    BriskodaChunkConfig,
    build_briskoda_chunks,
    chunk_manifest,
    count_tokens,
    load_briskoda_posts,
)

LIVE_CORPUS = Path(os.environ.get(
    'BRISKODA_JSONL',
    Path.home() / 'projects/docsmind-data/briskoda/full/superb_mk3.briskoda.jsonl',
))
LAB_DIR = Path.home() / 'projects/docsmind-data/briskoda/experiments/chunking-v1'
SNAPSHOT_DIR = LAB_DIR / 'snapshot'
PREVIEW_DIR = LAB_DIR / 'preview'
SNAPSHOT_PATH = SNAPSHOT_DIR / 'superb_mk3.briskoda.jsonl'

print(f'Repository: {ROOT}')
print(f'Live corpus: {LIVE_CORPUS}')
print(f'Lab output: {LAB_DIR}')
print(f'Live corpus exists: {LIVE_CORPUS.exists()}')


## 1. Freeze Corpus v1

A crawler output is an append-only source. An experiment should use an immutable snapshot so a model comparison tomorrow uses exactly the same posts as today.

This is corpus versioning. It separates **what data we evaluated** from **which embedding/index configuration we evaluated**.


In [ ]:
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

if not SNAPSHOT_PATH.exists():
    shutil.copy2(LIVE_CORPUS, SNAPSHOT_PATH)
    print('Created frozen snapshot.')
else:
    print('Reusing existing snapshot. Delete it manually only when you intend to create a new corpus version.')

sha256 = hashlib.sha256(SNAPSHOT_PATH.read_bytes()).hexdigest()
print(f'Snapshot size: {SNAPSHOT_PATH.stat().st_size / 1_000_000:.2f} MB')
print(f'SHA-256: {sha256}')


## 2. Validate and profile the source posts

Validation catches malformed JSON, missing citation fields, and duplicate `post_id` values before those errors become duplicate or uncitable vectors.

The length distribution answers a structural question: should most posts remain whole, or should most posts be split?


In [ ]:
posts = load_briskoda_posts(SNAPSHOT_PATH)
for post in posts:
    post['_token_count'] = count_tokens(post['text'])

lengths = pd.Series([post['_token_count'] for post in posts], name='tokens')
profile = pd.DataFrame({
    'metric': ['posts', 'topics', 'p50', 'p75', 'p90', 'p95', 'p99', 'max'],
    'value': [
        len(posts),
        len({post['topic_id'] for post in posts}),
        int(lengths.quantile(0.50)),
        int(lengths.quantile(0.75)),
        int(lengths.quantile(0.90)),
        int(lengths.quantile(0.95)),
        int(lengths.quantile(0.99)),
        int(lengths.max()),
    ],
})
display(profile)

thresholds = [40, 128, 256, 512, 700, 1024, 2048, 8192]
display(pd.DataFrame([
    {
        'threshold': threshold,
        'posts_above': int((lengths > threshold).sum()),
        'percentage': f'{(lengths > threshold).mean():.2%}',
    }
    for threshold in thresholds
]))


### Interpret the numbers before changing settings

A forum can contain enormous threads while most individual posts remain small. Retrieval should not embed a 500-page topic as one vector. It should index focused evidence and expand the surrounding conversation only after retrieval.

Long-context support is still useful for outliers, but it is not a reason to make every retrieval unit large.


## 3. Configure forum-aware chunking

The initial policy is:

1. Keep a post whole when the topic title plus post fits within 700 tokens.
2. When a reply is shorter than 40 tokens, prepend up to 300 tokens from the preceding post.
3. Split longer posts at sentence boundaries with a 700-token target and 100-token overlap.
4. Detect very large VCDS/diagnostic dumps. Keep every chunk for BM25, but allow only 8 evenly spaced chunks into dense embedding.
5. Repeat the topic title in every chunk.
6. Preserve stable IDs and the current post URL for citations.

These are hypotheses, not universal constants. Later retrieval evaluation decides whether they are good.


In [ ]:
# Change these values and rerun the remaining cells to compare policies.
chunk_config = BriskodaChunkConfig(
    max_tokens=700,
    overlap_tokens=100,
    short_post_tokens=40,
    previous_context_tokens=300,
)
chunk_config


In [ ]:
chunks = build_briskoda_chunks(posts, chunk_config)
manifest = chunk_manifest(posts, chunks, chunk_config)
display(pd.DataFrame([
    {'metric': key, 'value': value}
    for key, value in manifest.items()
    if key not in {'strategy_counts', 'config'}
]))
display(pd.DataFrame(
    manifest['strategy_counts'].items(),
    columns=['strategy', 'chunks'],
))

chunk_lengths = pd.Series([chunk['token_count'] for chunk in chunks])
print('Chunk token percentiles:')
display(chunk_lengths.quantile([0.50, 0.75, 0.90, 0.95, 0.99, 1.0]).to_frame('tokens'))
print(f'Duplicate chunk IDs: {len(chunks) - len({chunk["id"] for chunk in chunks})}')


### Route diagnostic dumps instead of deleting them

A VCDS scan is excellent for exact lookup, so `index_lexical=True` keeps all of it for BM25. Hundreds of nearly repetitive dense vectors can dominate semantic search, so only a stable, evenly spaced sample receives `index_dense=True`. The raw source post remains untouched.


In [ ]:
diagnostic_rows = []
for post_id in sorted({chunk['post_id'] for chunk in chunks if chunk['strategy'] == 'diagnostic_dump_split'}):
    post_chunks = [chunk for chunk in chunks if chunk['post_id'] == post_id]
    diagnostic_rows.append({
        'post_id': post_id,
        'topic': post_chunks[0]['topic_title'][:80],
        'BM25 chunks retained': len(post_chunks),
        'dense chunks embedded': sum(chunk['index_dense'] for chunk in post_chunks),
        'dense vectors avoided': sum(not chunk['index_dense'] for chunk in post_chunks),
    })
display(pd.DataFrame(diagnostic_rows))


### Compare the short-reply threshold: 20 vs 40 tokens

This comparison measures cost and how often context is borrowed. It cannot tell us which policy retrieves better answers; the labelled retrieval evaluation will decide that.


In [ ]:
policy_rows = []
for short_threshold in (20, 40):
    candidate_config = BriskodaChunkConfig(
        max_tokens=700, overlap_tokens=100,
        short_post_tokens=short_threshold, previous_context_tokens=300,
    )
    candidate_chunks = build_briskoda_chunks(posts, candidate_config)
    candidate_manifest = chunk_manifest(posts, candidate_chunks, candidate_config)
    policy_rows.append({
        'short threshold': short_threshold,
        'total chunks': len(candidate_chunks),
        'short replies with context': candidate_manifest['strategy_counts'].get('short_with_previous', 0),
        'dense-eligible chunks': candidate_manifest['dense_eligible_chunks'],
        'total tokens sent to BM25': sum(chunk['token_count'] for chunk in candidate_chunks),
        'tokens sent to embeddings': sum(chunk['token_count'] for chunk in candidate_chunks if chunk['index_dense']),
    })
display(pd.DataFrame(policy_rows))


## 4. Inspect examples—not just aggregate numbers

Metrics can say every chunk is under budget while the actual text is confusing. Inspect examples from every strategy. Ask:

- Does the chunk make sense without opening the full thread?
- Did borrowed context clarify the short reply?
- Is the citation URL for the post that supplies the answer?
- Does overlap repeat too much text?
- Would a real owner phrase a query that should retrieve this chunk?


In [ ]:
random.seed(7)
by_strategy = {}
for strategy in sorted({chunk['strategy'] for chunk in chunks}):
    candidates = [chunk for chunk in chunks if chunk['strategy'] == strategy]
    by_strategy[strategy] = random.sample(candidates, min(2, len(candidates)))

for strategy, examples in by_strategy.items():
    display(Markdown(f'## Strategy: `{strategy}`'))
    for example in examples:
        display(Markdown(
            f"**ID:** `{example['id']}`  \n"
            f"**Tokens:** {example['token_count']}  \n"
            f"**Citation:** {example['post_url']}  \n\n"
            f"```text\n{example['text'][:2500]}\n```"
        ))


## 5. Inspect specific difficult cases

The longest posts and shortest replies are where chunking policies usually break. The following cells surface those cases deliberately.


In [ ]:
longest = sorted(chunks, key=lambda chunk: chunk['token_count'], reverse=True)[:10]
display(pd.DataFrame([
    {
        'id': chunk['id'],
        'strategy': chunk['strategy'],
        'tokens': chunk['token_count'],
        'topic': chunk['topic_title'][:70],
        'url': chunk['post_url'],
    }
    for chunk in longest
]))

short_context_examples = [
    chunk for chunk in chunks if chunk['strategy'] == 'short_with_previous'
][:20]
display(pd.DataFrame([
    {
        'id': chunk['id'],
        'tokens': chunk['token_count'],
        'topic': chunk['topic_title'][:70],
        'preview': chunk['text'][-180:].replace('\n', ' '),
    }
    for chunk in short_context_examples
]))


## 6. Export a reviewable preview and manifest

The preview is intentionally small. It exists for human review and Git-independent experimentation, not as the final embedding input. The manifest records which corpus and settings produced it.


In [ ]:
preview = []
for strategy in sorted({chunk['strategy'] for chunk in chunks}):
    candidates = [chunk for chunk in chunks if chunk['strategy'] == strategy]
    preview.extend(random.sample(candidates, min(25, len(candidates))))

preview_path = PREVIEW_DIR / 'chunks-preview.jsonl'
with preview_path.open('w', encoding='utf-8') as handle:
    for chunk in preview:
        handle.write(json.dumps(chunk, ensure_ascii=False) + '\n')

manifest.update({
    'corpus_version': 'briskoda-superb-mk3-v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'source_path': str(SNAPSHOT_PATH),
    'source_sha256': sha256,
    'crawl_complete': False,
    'preview_records': len(preview),
})
manifest_path = LAB_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print(f'Preview: {preview_path}')
print(f'Manifest: {manifest_path}')
display(pd.DataFrame([manifest]))


## 7. Shape of the future AWS OpenSearch document

AWS Vector Ingestion accepts JSONL or Parquet from S3, but the vector ingestion job expects the vector field to already contain floating-point values. Our local embedding step must run before the S3 upload.

The same record keeps `text` for BM25 and metadata for filtering and citations. Only records with `index_dense=True` receive an `embedding` vector. For the full export, Parquet is preferable because numeric vectors are much more compact than JSON text.


In [ ]:
def aws_document_preview(chunk):
    return {
        'id': chunk['id'],
        'text': chunk['text'],
        'topic_id': chunk['topic_id'],
        'post_id': chunk['post_id'],
        'post_number': chunk['post_number'],
        'posted_at': chunk['posted_at'],
        'source_url': chunk['post_url'],
        'strategy': chunk['strategy'],
        'index_dense': chunk['index_dense'],
        'index_lexical': chunk['index_lexical'],
        # Add this field after model selection only when index_dense=True:
        'embedding': '<float array, or omitted for lexical-only chunks>',
    }

display(aws_document_preview(chunks[0]))


## 8. Questions for you to answer after inspection

Write your observations below or in a separate note:

1. Do short replies become clearer when the previous post is included?
2. Did you find cases where the opening question is also required?
3. Do 700-token guide chunks contain one coherent repair step or several unrelated steps?
4. Is 100-token overlap visibly repetitive?
5. Which metadata fields would you filter on in a real application?
6. What questions should enter the labelled retrieval evaluation set?

The next notebook should benchmark **BGE-small as the baseline**, **GTE ModernBERT as the efficient candidate**, and **Qwen3-Embedding-0.6B as the quality candidate** on the same frozen chunks. The decision will use retrieval quality, latency, memory, throughput, and index size—not model reputation.


In [ ]:
# Add questions you believe a real Superb owner would ask.
QUESTIONS_FOR_EVAL = [
    'Why does my DSG hesitate when the car is cold?',
    'How do I diagnose an intermittent KESSY ignition switch?',
    'What can cause DPF limp mode without a dashboard warning?',
]
QUESTIONS_FOR_EVAL
